# T7 It1: signal table (every candidate, every signal after fine matching)

Test_Iteration_7 works only on the decision after fine matching, **without multi-query agreement** (ruled out: it made wrong answers on fresh
NUDT city photos in Test_Iteration_6). This notebook re-fits the stored LoFTR matches of every candidate (27 queries x M = 10) of every photo of
the five seen sets and saves one table per set (block `train/checks/decision_signals.py`). The blind set (NUDT fresh 606) is **not** touched here.

Signals (reasons in the block's docstring): pipeline similarity inliers (`inl_s3`), LO-RANSAC re-fit (`inl_lo`), translation consensus at 2 / 3 px
(`inl_t2`, `inl_t3`) and at confidence 0.15 (`inl_t3_15`), guided 0.15 lines (`guided15`), a-contrario log10 NFA (`lnfa_t3`, `lnfa_s3`), coverage
(`cover9`, `hull`, `spread`), fit plausibility (`scale_dev`, `angle`, `shift_px`), `resid`, `conf_inl`, `hc_ratio`, plus stored `z`, `coarse`, `sharp`, `rank`.

In [ ]:
import os, sys, time
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import pandas as pd
from train.checks.decision_signals import build_signal_table
DUMPS = ROOT / "tests/Test_Iteration_6/results/rich_dump"
OUT = ROOT / "tests/Test_Iteration_7/results/signals"
OUT.mkdir(parents=True, exist_ok=True)
SETS = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404", "nudt505": "nudt_fresh505", "nudt606": "nudt_fresh606"}
ONLY = os.environ.get("ONLY")                     # e.g. ONLY=nudt606: the blind set, built after the rules were frozen
SETS = {k: v for k, v in SETS.items() if (k == ONLY if ONLY else k != "nudt606")}
WORKERS = int(os.environ.get("WORKERS", "12"))

In [ ]:
summary = []
for name, folder in SETS.items():
    t = time.time()
    table = build_signal_table(DUMPS / folder, name, WORKERS)
    table.to_parquet(OUT / f"{name}.parquet") if False else table.to_csv(OUT / f"{name}.csv.gz", index=False)
    same = (table.inliers == table.inl_s3).mean()
    summary.append(dict(set=name, photos=table.photo.nunique(), candidates=len(table), seconds=round(time.time() - t, 1), refit_equals_stored=round(same, 4)))
    print(summary[-1])
pd.DataFrame(summary)

The re-fit reproduces the stored inliers on more than 99 % of candidates; the rest differ by 1-2 inliers (RANSAC sampling order). Every later
notebook reads `results/signals/<set>.csv.gz`.